# Can the subcomponents copy a finished fine-tune, and how sure is the check that says so?

*E34, `experiments/teacher_training`, 2026-10-23 to 2026-10-26. Record: `reports/analyses/2026-10-23_analysis_teacher-training.md`,
with one section per amendment. This notebook gives E34 and Amendments 1 to 4 in brief, then Amendment 5 in full.*

## E34 so far

**Teacher training** trains offsets on the alive subcomponents' read and write vectors so that the decomposed model
copies a fine-tuned model's predictions, with Γ and Δ frozen. The loss is the KL from the fine-tuned model's
next-token distribution to the student's, so it needs no task labels. The fine-tune left wholly in Δ, the
*do-nothing* state, is the reference.

- **E34.** It works on all three trigger tasks tried. Trained on the task's own text, every copy kept the behaviour and
  passed the faithfulness check, including where the adjoint split had lost the behaviour. Trained on ordinary Pile text
  alone, every copy also got there by the last checkpoint, including the emoticon task, where we had predicted failure.
- **Amendment 1.** The rare trigger positions in the Pile text did not teach the emoticon copy: with every row holding
  a trigger replaced, the copies still reached a target probability of 0.94 and 0.89. The copy's weight change is not
  the fine-tune's.
- **Amendment 2.** Removing also the rows where the trigger is partly on lowered both Pile-text copies' end state, to
  0.87 and 0.84.
- **Amendment 3.** A rank-8 LoRA is copied as well from the task's text, but only partly from Pile text alone.
- **Amendment 4.** A second batch order moved a copy's end state by at most 0.03, which makes Amendment 2's falls real.
  Under an adversary eight times longer every copy stayed at the unchanged model's level, while five of the six
  fine-tunes did not. It also found that two versions of the adversarial check read one fine-tune six times apart.
  Amendment 5 asks why.

## Amendment 5: why do the two adversaries disagree, and is a copy unique?

### The answer

The two adversaries disagree because they start from different random points. Rows, readout precision and step budget
change little. On the r1 SGD fine-tune left in Δ, the budget code reads about 9 times the unchanged model from its own
start. Started where the harness starts, it reads about 1.7, as the harness does. The unchanged model reads nearly the
same from either start. So a one-start verdict on a fine-tune can understate the worst case several-fold. Separately,
two Pile-text copies at two batch orders behave alike, but their weight changes point well apart.

### What we were trying to test

Amendment 4 read one state twice. E34's harness read the r1 SGD teacher's do-nothing state, the fine-tune $\delta W$
left in Δ, at 1.59 times the unchanged model $W_0$. The budget sweep read the same state at 9.24, with the same 20
steps. Every faithfulness verdict in the project rests on one of these two measures, so the gap has to be located
before either is used on the larger model.

**The search both run.** It looks for the masks that make the masked model's predictions differ most from the
state's own. Each subcomponent gets a source $a \in [0, 1]$ and the mask $g + (1 - g)\,a$, with $g$ its causal
importance from Γ. Δ's mask is one more source. One set of sources serves every row. The search starts from random
sources and takes sign steps up the gradient of the mean KL. The reading is the KL in nats per position under the
final masks. The ratio divides it by $W_0$'s reading, so 1 means the state is no easier to break than the unchanged
model.

- **The harness** is the authors' code. It searches two blocks of 128 Pile rows separately and averages them. It
  takes 20 steps of 0.1 and returns its own last objective, computed in bf16.
- **The budget code** is the same search in our code, on the first block only. It takes $n$ steps of $2/n$ and
  reads the KL afterwards in a separate full-precision pass.

Reading the two codes side by side left four differences that could matter: the rows, the readout, the budget and the
start. Both draw their random sources after the same seed. But the harness hands the draws to the matrices in the
model's order, and the budget code in sorted order. So they start from different, equally random numbers. Each reading
below changes one difference, at $W_0$ and at the teacher, adversary seed 0. An identity check runs our code from the
harness's start with the harness's readout. It should give the harness's own number.

**Is a copy unique?** The task-text copy and the Pile-text copy of the same teacher were each retrained at batch
orders 0 and 1. For each pair we read the cosine between the two weight changes over all 24 matrices. A cosine of 1 is
the same direction and 0 is unrelated; a random direction reads about 0.0002.

**Controls, run first.** $W_0$ had to read its known adversary-seed floor. The regenerated teacher had to reproduce
E34's and Amendment 4's readings before any variant was read.

### What we expected, and why

- **The rows do not carry the gap.** Adding the second block, or splitting the harness's blocks, moves the teacher's
  ratio by under 25%, the copies' batch-order spread.
- **One difference in the code path carries it**, the readout or the start. We filed no guess on which; Amendment 4's
  report had suspected the rows or the readout.
- **At $W_0$ every variant reads within 10% of its default**, at high confidence. The two measures had agreed on
  $W_0$ within 0.04 nats.
- **The copies at two batch orders point the same way**, cosine at least 0.8 for both arms, at medium confidence.
  Their target probabilities had moved by under 0.005. Below 0.5, by the filed rule, a copy is one of many weight
  changes with the same behaviour.

### Is the gap in the rows, the readout, the budget, or the start?

![amend5](../experiments/teacher_training/figures/amend5.png)

**Reading it.** The start carries all of it. The figure plots the KL each reading reaches, hollow at the unchanged
model and filled at the teacher, grouped by the start the search took. From the harness's start the teacher reads
about 1.1 to 1.3 nats, or 1.6 to 1.7 times the unchanged model. That holds whichever code runs the search, which rows
it sees, which readout is used and how many steps it takes. From the budget code's start the teacher reads 6 to 7
nats, or 9 to 10 times. The readout moved the teacher's ratio by 0.2%. Adding the second block to one search moved it
by 10%. Eight times as many steps raised each start's reading by about a fifth and left the gap at six times. The
identity check passed: from the harness's start, our code read the harness's number within 4%. The caveat: these are
two random starts on one state at one seed, so we cannot say which reading is typical.

| reading | start | teacher, nats | ratio to $W_0$ |
|---|---|---|---|
| harness, rows 1-128 | harness | 1.084 | 1.60 |
| budget code from the harness's start | harness | 1.128 | 1.67 |
| harness, 160 steps | harness | 1.269 | 1.61 |
| budget code | budget | 6.130 | 9.24 |
| budget code, harness readout | budget | 6.124 | 9.22 |
| budget code, all 256 rows | budget | 7.040 | 10.19 |
| budget code, 160 steps | budget | 7.391 | 10.45 |

### Does the start matter at the unchanged model?

**Reading it.** Hardly at 20 steps, and far less than at the teacher. In the figure above the hollow points sit in a
narrow band. At 20 steps on the first block, $W_0$ reads within 2.4% from either start. At the teacher the same two
starts differ five-fold. Seven of the eight filed comparisons at $W_0$ agree within 4.3%. The one miss is at 160
steps, where the harness's start reads 0.786 nats and the budget code's 0.708. That is 11% apart, against the filed
10%. The two readings differ only in the start. The budget code's own spread between two adversary seeds at 160 steps
was 23%, so this miss sits inside known noise.

### Does a second batch order give the same copy?

![amend5_cos](../experiments/teacher_training/figures/amend5_cos.png)

**Reading it.** Nearly, for the task-text copy; not for the Pile-text copy. The two task-text copies have a cosine of
0.91, about 24 degrees apart. The two Pile-text copies have a cosine of 0.43, about 65 degrees apart. That is below
the filed 0.5, so by the filed rule a Pile-text copy is one of many weight changes with the same behaviour. Their
behaviour is the same: held-out target probability 0.974 at both batch orders. Each lines up with the fine-tune's
$\delta W$ equally, at about 0.17. A random direction reads at most 0.0002. The caveat: the cosine is on the total
weight change, so two copies that spread one change differently over the subcomponents would read 1. That blind spot
can only hide a difference, so the two placements are at most this alike.

### Did the controls behave?

All passed. The unchanged model read an adversary-seed floor of 0.0748 against 0.075. The regenerated teacher read a
target probability of 0.983 and a size of 0.0287, as in E34. Its harness ratio was 1.588 and its budget ratio 9.24, as
in E34 and Amendment 4. At zero steps the two codes, from the same start, agreed to $4 \times 10^{-8}$.

## What we learned

The disagreement Amendment 4 found is not between two measures but between two random starts of one measure. On this
fine-tune left in Δ, one start finds masks that raise the KL to 6 nats and the other stops near 1. An eight times
longer search did not close the gap. Both are valid searches, so each reading is only a lower bound on the worst
masks. A verdict from one start can therefore be lenient: the harness read this teacher inconclusive at 1.59, where
the other start reads 9.2. The unchanged model barely notices the start, so the sensitivity belongs to this state.
Which other verdicts it touches was not tested. On E37's broad fine-tune the two starts agreed, and Amendment 4's
copies passed from both. For the larger model, a Pile-text copy is one of several weight changes with the same
behaviour. A claim about where such a copy puts the update needs two batch orders behind it.

## How compelling is the evidence

- **Two random starts, one adversary seed, one state.** How readings spread over many starts, and on which states, is
  unmeasured.
- **The identity check closes the code path.** At zero steps the codes agreed to $4 \times 10^{-8}$, and after 20
  steps within 4%. No other code difference is hiding.
- **Noise.** The harness's adversary-seed floor at $W_0$ is 0.075. The budget code's two-seed spread is 7% at 20
  steps and 23% at 160. A five- to six-fold gap is far outside both.
- **One teacher and one pair of batch orders for the copies.** E37's copies, trained on their teacher's own rows,
  read 0.93 between batch orders.

## Surprises

- **Reversed: the Pile-text copies at two batch orders point 65 degrees apart.** We predicted a cosine of at least 0.8
  on both arms, at medium confidence. The Pile-text pair read 0.43, which the filed rule reads as "one of many". Their
  target probabilities agree to 0.0004. Why they differ was not tested.
- **Missed narrowly: at $W_0$, 160 steps from the two starts read 11% apart**, against the filed 10%.
- **A random start carries a six-fold gap.** Amendment 4's report had suspected the rows or the readout.
- **The task-text copies line up with $\delta W$ at about 0.25**, a little above the plan's expected 0.08 to 0.22.

## What next

1. **For the user's decision: read every adversarial verdict as the maximum over several random starts.** The
   harness's verdicts so far rest on one start. Each further start costs one more search, about 25 s per block on an
   H100. If passing states still pass under the maximum, those passes stand. If not, they are re-read before the
   larger-model run uses the harness.
2. **How many starts.** Read $W_0$ and this teacher from about eight starts, to see whether 1.6 or 9.2 is typical.
3. **The larger-model write-up states that a Pile-text copy is not unique.** A third batch order or a per-matrix
   cosine would say whether the two copies differ everywhere or in a few matrices.

## Scorecard, Amendment 5

Filed 2026-10-26 20:03:20 in `experiments/teacher_training/results/predictions_amend5.json`, sha256 `f447e43f…`,
before the run. Scored by the run's own `amend5.score`.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | the unchanged model reads its known values | floor 0.075 within 5% | 0.0748 | pass |
| G1 | the teacher reproduces E34 and Amendment 4 | P, size, harness and budget ratios | 0.983, 0.0287, 1.588, 9.24 | pass |
| P1 | the rows do not carry the gap | rows256 and harness_b0 within 25% of their defaults | +10%, +0.4% | **holds** |
| P2 | one code-path difference carries it | budget.bf16 or start within 25% of the other's ratio | start 1.67 near the harness's 1.59; both implicate the start | **holds** |
| P3 | at $W_0$ every variant within 10% | eight pairs | harness160_b0 +11.1% against budget160 | **fails**, narrowly |
| P4 | copies at two batch orders point the same way | cosine ≥ 0.8 on both arms | 0.913 and 0.429 | **fails**: "one of many" |

## Not run

More than two random starts on any state; adversary seeds past 0; other adversary families. The harness at 160 steps
on its second block; our code from the harness's start at 160 steps. Any other teacher under both starts, including
the r0 teachers the harness passed. Harness or budget readings of the four new copies; a third batch order; per-matrix
cosines.

## Provenance

- RunPod H100 80GB HBM3, pod 01M5WYADKPDN6TRMBFYD0KRBV3. Run 20:28 to 20:52, exit 0.
- The pod sat idle until 21:34 while its runner waited at a permission prompt. The lead terminated it after sha256
  checks. Session spend $5.82 for E37 and this amendment together.
- Results `experiments/teacher_training/results/teacher_training_amend5.json`, sha256 `1454bfb4…`. Log
  `results/large/run_amend5.log`, `f18a527f…`. Copies `results/large/r1_t534_v369/amend5_*_step1000.pt`.
- Code `experiments/teacher_training/amend5.py`. Figures `figures/amend5.png` and `figures/amend5_cos.png` from
  `plot_amend5.py`.